# 🔬 Entrenamiento Final YOLO11 - Bromatología UTEQ
### Proyecto: Asistente Inteligente de Laboratorio con Visión Artificial y RAG
**Universidad Técnica Estatal de Quevedo (UTEQ)**  
**Facultad de Ciencias Pecuarias y Biológicas**

---

Este cuaderno entrena **YOLO11 Nano (`yolo11n.pt`)** utilizando el dataset etiquetado manualmente en **Roboflow**, garantizando la máxima precisión en las coordenadas y un ajuste perfecto a los equipos.

## 🛠️ Paso 1: Configurar GPU y Dependencias
Verifica que estés conectado con **GPU T4** (`Entorno de ejecución > Cambiar tipo de entorno de ejecución > GPU T4`).

In [ ]:
# Verificar GPU
!nvidia-smi

# Instalar Ultralytics
!pip install -q --upgrade ultralytics

import ultralytics
ultralytics.checks()

## 📦 Paso 2: Subir el Dataset exportado desde Roboflow (`.zip`)
Ejecuta esta celda y selecciona el archivo `.zip` que descargaste de Roboflow.

In [ ]:
import os
import zipfile
from google.colab import files

# Limpiar entorno anterior
!rm -rf dataset runs *.zip

print("Sube el archivo .zip descargado de Roboflow:")
uploaded = files.upload()
zip_name = list(uploaded.keys())[0]

print(f"Descomprimiendo {zip_name}...")
os.makedirs('dataset', exist_ok=True)
with zipfile.ZipFile(zip_name, 'r') as zip_ref:
    zip_ref.extractall('dataset')

# Buscar el archivo data.yaml generado por Roboflow
import glob
yaml_files = glob.glob('dataset/**/data.yaml', recursive=True)
if yaml_files:
    data_yaml_path = yaml_files[0]
    print(f"✅ Dataset descomprimido y data.yaml encontrado en: {data_yaml_path}")
    
    # Corregir rutas absolutas en el YAML para que apunten al entorno de Colab
    with open(data_yaml_path, 'r') as f:
        yaml_content = f.read()
    
    # Roboflow a veces exporta rutas absolutas rotas, las forzamos a relativas
    import re
    yaml_content = re.sub(r'train: .*', 'train: train/images', yaml_content)
    yaml_content = re.sub(r'val: .*', 'val: valid/images', yaml_content)
    yaml_content = re.sub(r'test: .*', 'test: test/images', yaml_content)
    
    with open(data_yaml_path, 'w') as f:
        f.write(yaml_content)
else:
    print("❌ Error: No se encontró data.yaml en el ZIP.")

## 🚀 Paso 3: Entrenar YOLO11 Nano (`yolo11n.pt`)
Entrenaremos durante 100 épocas. El proceso tomará unos 5-8 minutos.

In [ ]:
from ultralytics import YOLO

# Cargar modelo base
model = YOLO('yolo11n.pt')

# Iniciar entrenamiento usando el YAML de Roboflow
results = model.train(
    data=data_yaml_path,  # Ruta al data.yaml que corregimos arriba
    epochs=100,           # 100 épocas para asegurar convergencia perfecta
    imgsz=640,
    batch=16,
    patience=25,          # Early stopping si no mejora en 25 épocas
    save=True,
    device=0,
    project='yolo11_uteq_final',
    name='train_roboflow',
    exist_ok=True,
    # Ligeras aumentaciones para robustecer el modelo
    mosaic=1.0,
    degrees=5.0,
    scale=0.3
)

print("🎉 ¡Entrenamiento Final completado!")

## 📊 Paso 4: Evaluar Métricas (mAP)

In [ ]:
metrics = model.val()
print(f"mAP@0.5: {metrics.box.map50:.4f}")

## 📱 Paso 5: Exportar a TensorFlow Lite y Descargar

In [ ]:
import glob
import shutil
from google.colab import files
from ultralytics import YOLO

# Cargar el mejor modelo entrenado
best_weights = glob.glob('/content/yolo11_uteq_final/train_roboflow/weights/best.pt')[0]
print(f"Exportando modelo: {best_weights}")
best_model = YOLO(best_weights)

# Exportar a TFLite
exported_path = best_model.export(format='tflite', imgsz=640)

# Renombrar y descargar
target_tflite = 'yolo11_bromatologia.tflite'
tflite_files = glob.glob('/content/**/*.tflite', recursive=True)
shutil.copyfile(tflite_files[0], target_tflite)

print(f"✅ Descargando archivo final ({os.path.getsize(target_tflite)/(1024*1024):.2f} MB)...")
files.download(target_tflite)
print("Guárdalo en: DetectorDeMaterialesLaboratorio/app/src/main/assets/")